In [1]:
"""
08_random_forest.py

Same setup as 07_logistic_regression.py, but with a random forest.
Train/test split is by bookyear (2018 train, 2019 test) - see
utils/time_based_split.py. See 11_train_window_sensitivity.ipynb for
the check on whether adding 2017 as a second training year helps.
"""


'\n08_random_forest.py\n\nSame setup as 07_logistic_regression.py, but with a random forest.\nTrain/test split is by bookyear (2018 train, 2019 test) - see\nutils/time_based_split.py. See 11_train_window_sensitivity.ipynb for\nthe check on whether adding 2017 as a second training year helps.\n'

In [2]:
from utils.load_data_features import load_data_features

df = load_data_features()


In [3]:
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

from config import FEATURES_FINANCIAL, RANDOM_STATE, TARGET, WINSOR_COLUMNS
from utils.model_results import save_model_results
from utils.print_section import print_section
from utils.time_based_split import time_based_split
from utils.winsorizer import Winsorizer

MODEL_NAME = "Random Forest"

X_train, X_test, y_train, y_test = time_based_split(df, FEATURES_FINANCIAL, TARGET)

pipeline = Pipeline([
    ("winsorizer", Winsorizer(columns=WINSOR_COLUMNS)),
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(
        n_estimators=500,
        random_state=RANDOM_STATE,
        class_weight="balanced_subsample",
        n_jobs=-1,
    )),
])

pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)
y_prob = pipeline.predict_proba(X_test)[:, 1]

print_section("Performance")

metrics = {
    "accuracy": accuracy_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred, zero_division=0),
    "recall": recall_score(y_test, y_pred),
    "f1": f1_score(y_test, y_pred),
    "roc_auc": roc_auc_score(y_test, y_prob),
    "pr_auc": average_precision_score(y_test, y_prob),
}

for name, value in metrics.items():
    print(f"{name:10s}: {value:.4f}")

print_section("Confusion matrix")

cm = confusion_matrix(y_test, y_pred)
print(cm)

tn, fp, fn, tp = cm.ravel()
print()
print(f"True Negatives : {tn:,}")
print(f"False Positives: {fp:,}")
print(f"False Negatives: {fn:,}")
print(f"True Positives : {tp:,}")

print_section("Feature importance")

importances = pd.DataFrame({
    "feature": FEATURES_FINANCIAL,
    "importance": pipeline.named_steps["model"].feature_importances_,
}).sort_values(by="importance", ascending=False)

print(importances)

print_section("Highest predicted probabilities")

prob_df = pd.DataFrame({"actual": y_test.values, "probability": y_prob})
print(prob_df.sort_values(by="probability", ascending=False).head(20))

save_model_results(MODEL_NAME, metrics)



Performance
accuracy  : 0.9977
precision : 0.0155
recall    : 0.0032
f1        : 0.0053
roc_auc   : 0.7576
pr_auc    : 0.0107

Confusion matrix
[[324363    127]
 [   617      2]]

True Negatives : 324,363
False Positives: 127
False Negatives: 617
True Positives : 2

Feature importance
         feature  importance
2       solvency    0.248704
0  profitability    0.228484
1      liquidity    0.175115
5           size    0.154895
3      structure    0.102114
4        log_age    0.090688

Highest predicted probabilities


        actual  probability
235095       0     0.793299
237119       0     0.793299
234775       0     0.793299
225628       0     0.793299
229532       0     0.793299
236524       0     0.793299
232572       0     0.793299
235094       0     0.793299
228470       0     0.793299
236449       0     0.793299
233441       0     0.793299
228508       0     0.793299
237790       0     0.793299
229958       0     0.793299
235268       0     0.793299
226340       0     0.793299
237811       0     0.793299
230923       0     0.793299
237405       0     0.793299
236547       0     0.793299
